# Week 1 Lab — Acquiring, Loading, and Visualizing Geodata

**Course:** Geo-Data and Geo-Data Management  
**Lab focus:** acquire several forms of geodata, load them in Python, and visualize what was returned.


## What you will do today

By the end of this notebook, you should be able to:

1. run the notebook using the **IT_Geo** kernel;
2. acquire geodata from a web service or file URL;
3. load four different data forms in Python;
4. create one simple visualization for each example.

We repeat the same workflow:

```text
acquire → load → inspect → visualize
```

The examples are **vector polygons**, **raster imagery**, **trajectory data**, and **3D point-cloud data**.


## Before you start

Make sure the notebook is using the **IT_Geo** kernel. Run the cells from top to bottom.

You do **not** need to memorize the Python syntax today. Focus on what each dataset looks like after it is loaded and visualized.


# 1. Import the packages

We use `requests`, `pandas`, `geopandas`, `rasterio`, and `matplotlib`.


In [ ]:
from io import StringIO
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

data_dir = Path("week1_data")
data_dir.mkdir(exist_ok=True)

print("Packages imported successfully.")


# 2. Vector data — Vienna district boundaries

We retrieve a polygon layer from the **City of Vienna Web Feature Service (WFS)**.

- **Source:** City of Vienna
- **Information type:** geometry + attributes
- **Representation:** polygons
- **Storage / access:** WFS → JSON


In [ ]:
vienna_wfs_url = (
    "https://data.wien.gv.at/daten/geo"
    "?service=WFS&request=GetFeature&version=1.1.0"
    "&typeName=ogdwien:BEZIRKSGRENZEOGD"
    "&srsName=EPSG:4326&outputFormat=json"
)

response = requests.get(vienna_wfs_url, timeout=30)
response.raise_for_status()
vector_json = response.json()

districts = gpd.GeoDataFrame.from_features(
    vector_json["features"], crs="EPSG:4326"
)

print("Rows:", len(districts))
print("CRS:", districts.crs)
print("Geometry:", districts.geometry.geom_type.unique())
districts.head()


In [ ]:
ax = districts.plot(figsize=(7, 7), edgecolor="black")
ax.set_title("Vienna district boundaries")
ax.set_axis_off()
plt.show()


# 3. Raster data — GeoTIFF imagery

We download a small GeoTIFF from the Rasterio example data.

- **Source:** Rasterio example dataset
- **Information type:** image-band values
- **Representation:** raster grid
- **Storage / access:** GeoTIFF file


In [ ]:
raster_url = (
    "https://raw.githubusercontent.com/rasterio/rasterio/"
    "main/tests/data/RGB.byte.tif"
)
raster_path = data_dir / "RGB.byte.tif"

response = requests.get(raster_url, timeout=60)
response.raise_for_status()
raster_path.write_bytes(response.content)

with rasterio.open(raster_path) as src:
    raster = src.read()
    raster_crs = src.crs
    raster_bounds = src.bounds

print("Shape (bands, rows, columns):", raster.shape)
print("CRS:", raster_crs)
print("Bounds:", raster_bounds)


In [ ]:
with rasterio.open(raster_path) as src:
    fig, ax = plt.subplots(figsize=(8, 6))
    show(src, ax=ax)
    ax.set_title("Raster imagery")
    plt.show()


# 4. Trajectory data — locations over time

We download a small trajectory sample distributed with MovingPandas.

- **Source:** GeoLife / MovingPandas sample
- **Information type:** location + time
- **Representation:** trajectory
- **Storage / access:** CSV file


In [ ]:
trajectory_url = (
    "https://raw.githubusercontent.com/movingpandas/movingpandas/"
    "main/tutorials/data/demodata_geolife.csv"
)
trajectory_path = data_dir / "demodata_geolife.csv"

response = requests.get(trajectory_url, timeout=60)
response.raise_for_status()
trajectory_path.write_bytes(response.content)

trajectories = pd.read_csv(trajectory_path, sep=";")
trajectories["t"] = pd.to_datetime(trajectories["t"])

print("Rows:", len(trajectories))
print("Trajectory IDs:", sorted(trajectories["trajectory_id"].unique()))
trajectories[["X", "Y", "t", "trajectory_id"]].head()


In [ ]:
trajectory_id = 1
one_trajectory = (
    trajectories[trajectories["trajectory_id"] == trajectory_id]
    .sort_values("t")
)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(one_trajectory["X"], one_trajectory["Y"], marker="o", markersize=2)
ax.set_title(f"Trajectory {trajectory_id}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


# 5. Point-cloud data — 3D points

We download a small ASCII PCD file from the Point Cloud Library sample-data repository.

- **Source:** Point Cloud Library sample dataset
- **Information type:** 3D positions
- **Representation:** point cloud
- **Storage / access:** PCD file


In [ ]:
pointcloud_url = (
    "https://raw.githubusercontent.com/PointCloudLibrary/data/"
    "master/tutorials/min_cut_segmentation_tutorial.pcd"
)
pointcloud_path = data_dir / "sample_point_cloud.pcd"

response = requests.get(pointcloud_url, timeout=60)
response.raise_for_status()
pointcloud_path.write_bytes(response.content)

lines = pointcloud_path.read_text(encoding="utf-8").splitlines()
data_start = next(
    i for i, line in enumerate(lines)
    if line.strip().lower() == "data ascii"
) + 1

point_cloud = pd.read_csv(
    StringIO("\n".join(lines[data_start:])),
    sep=r"\s+", names=["x", "y", "z"]
)

print("Number of 3D points:", len(point_cloud))
point_cloud.head()


In [ ]:
plot_points = point_cloud.sample(
    n=min(4000, len(point_cloud)), random_state=42
)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(plot_points["x"], plot_points["y"], plot_points["z"], s=2)
ax.set_title("3D point cloud")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.show()


# 6. Compare the examples

The same four questions from the lecture help us describe each dataset.


In [ ]:
comparison = pd.DataFrame([
    ["Vienna districts", "City of Vienna", "geometry + attributes", "polygons", "WFS → JSON"],
    ["Raster imagery", "Rasterio sample", "image-band values", "raster grid", "GeoTIFF"],
    ["Trajectory", "GeoLife / MovingPandas", "location + time", "trajectory", "CSV"],
    ["3D point cloud", "PCL sample", "3D positions", "point cloud", "PCD"],
], columns=["example", "source", "information_type", "representation", "storage_access"])

comparison


# 7. Lab checkpoint — Change one visualization

Choose **one** example and change one parameter:

- **Vector:** `vector_style = "boundary"`
- **Raster:** choose `raster_band = 1`, `2`, or `3`
- **Trajectory:** change `checkpoint_trajectory_id`
- **Point cloud:** change `point_sample_size`

You only need to edit the values in the next cell.


In [ ]:
checkpoint_choice = "trajectory"   # vector, raster, trajectory, or pointcloud

vector_style = "filled"
raster_band = 1
checkpoint_trajectory_id = 2
point_sample_size = 1500

print("Checkpoint choice:", checkpoint_choice)


In [ ]:
if checkpoint_choice == "vector":
    fig, ax = plt.subplots(figsize=(7, 7))
    if vector_style == "boundary":
        districts.boundary.plot(ax=ax)
    else:
        districts.plot(ax=ax, edgecolor="black")
    ax.set_title(f"Vector checkpoint — {vector_style}")
    ax.set_axis_off()
    plt.show()

elif checkpoint_choice == "raster":
    with rasterio.open(raster_path) as src:
        band = src.read(raster_band)
        fig, ax = plt.subplots(figsize=(8, 6))
        show(band, ax=ax)
        ax.set_title(f"Raster checkpoint — band {raster_band}")
        plt.show()

elif checkpoint_choice == "trajectory":
    selected = trajectories[
        trajectories["trajectory_id"] == checkpoint_trajectory_id
    ].sort_values("t")
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(selected["X"], selected["Y"], marker="o", markersize=2)
    ax.set_title(f"Trajectory checkpoint — ID {checkpoint_trajectory_id}")
    plt.show()

elif checkpoint_choice == "pointcloud":
    selected = point_cloud.sample(
        n=min(point_sample_size, len(point_cloud)), random_state=42
    )
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(selected["x"], selected["y"], selected["z"], s=2)
    ax.set_title(f"Point-cloud checkpoint — {len(selected)} points")
    plt.show()

else:
    raise ValueError("Choose vector, raster, trajectory, or pointcloud")


## Complete the checkpoint notes

Double-click this Markdown cell and replace the placeholders below.

**1. Which data form did you choose?**  
YOUR ANSWER

**2. What parameter did you change?**  
YOUR ANSWER

**3. What changed in the visualization?**  
YOUR ANSWER

**4. Describe the dataset in one line:**  
Source = ... ; Information type = ... ; Representation = ... ; Storage / access = ...


## Checkpoint requirements

- [ ] All four original visualizations are visible.
- [ ] You changed one checkpoint parameter.
- [ ] Your checkpoint visualization is visible.
- [ ] You completed all four checkpoint notes.

Keep the code outputs visible in the notebook.


## Submit your notebook to Moodle

1. Save the notebook **after** your final run.
2. Check that your edited parameter, outputs, and checkpoint notes are visible.
3. Submit the completed `.ipynb` file to the Week 1 lab checkpoint on Moodle.
